In [17]:
import numpy as np
import random
import math

In [19]:
# --- Safe Operations ---

EPS = 1e-6  # small constant to avoid divide-by-zero and overflow

def safe_div(a, b):
    """Safe division: returns a / b if |b| > EPS, otherwise returns 0."""
    try:
        return a / (b if abs(b) > EPS else EPS)
    except Exception:
        return 0.

def safe_log(a):
    """Safe natural log: returns log(|a|) if a is not too close to zero, otherwise returns 0."""
    if abs(a) < EPS:
        return 0.
    return np.log(abs(a))

# --- Primitive Set ---

# Define a set of functions that can operate elementwise on numpy arrays
primitive_set = {
    'add': {'func': np.add, 'arity': 2, 'repr': '+'},
    'sub': {'func': np.subtract, 'arity': 2, 'repr': '-'},
    'mul': {'func': np.multiply, 'arity': 2, 'repr': '*'},
    'div': {'func': safe_div, 'arity': 2, 'repr': '/'},
    'sin': {'func': np.sin, 'arity': 1, 'repr': 'sin'},
    'cos': {'func': np.cos, 'arity': 1, 'repr': 'cos'},
    'tan': {'func': np.tan, 'arity': 1, 'repr': 'tan'},
    'exp': {'func': np.exp, 'arity': 1, 'repr': 'exp'},
    'log': {'func': safe_log, 'arity': 1, 'repr': 'log'},
    # Add other functions as needed...
}

# We also treat the input variables as terminals. In our case, since the shape of x is (n_vars, n_samples), we refer to them by index, e.g. x[0].
terminal_set = ['x[%d]' % i for i in range(10)]  # allocate up to 10 possible variables, will use as many as needed
# Optionally, add constants as terminals:
constant_pool = [str(round(random.uniform(-5, 5), 2)) for _ in range(5)]

In [21]:
# --- Expression Tree Structure ---

class Node:
    def __init__(self, content, children=None):
        self.content = content  # either a key for primitive_set or a terminal (variable or constant)
        self.children = children if children is not None else []

    def is_terminal(self):
        return len(self.children) == 0

    def evaluate(self, x):
        """Recursively evaluate the expression tree.
           x is expected to be a numpy array where each row is an input variable."""
        # If node is a terminal, interpret it.
        if self.is_terminal():
            # If the content is of form 'x[i]', extract the row.
            if isinstance(self.content, str) and self.content.startswith('x['):
                # Evaluate expression, e.g., "x[0]" returns the first row
                index = int(self.content[2:-1])
                return x[index]
            else:
                # It is a constant string, convert to float and return a numpy array of constant
                return float(self.content) * np.ones(x.shape[1])
        else:
            # Get the function from the primitive set.
            op = primitive_set[self.content]
            # Recursively evaluate children:
            args = [child.evaluate(x) for child in self.children]
            # Apply the operator in a vectorized way.
            return op['func'](*args)

    def __str__(self):
        """Return a string representation of the expression."""
        if self.is_terminal():
            return str(self.content)
        else:
            op_repr = primitive_set[self.content]['repr']
            if primitive_set[self.content]['arity'] == 1:
                return f"{op_repr}({self.children[0]})"
            elif primitive_set[self.content]['arity'] == 2:
                return f"({self.children[0]} {op_repr} {self.children[1]})"
            else:
                return f"{self.content}(" + ", ".join(str(child) for child in self.children) + ")"